# Banyan City — free anime rendering on Kaggle

Renders a node's `shots.md` prompts into per-beat clips with **AnimateDiff** on an anime-tuned
**SD1.5** checkpoint, on Kaggle's free GPU quota (30 h/week) — the tree's permanent $0 rendering
floor, reproducible by any citizen (**compute-as-watering**, see `WATERING.md`).

**Setup:** Kaggle → New Notebook → File → Import Notebook → this file. Settings: Accelerator =
**GPU T4 x2**, Internet = ON, phone-verified account. Or drive it headless from a laptop with
`python3 pipeline/kaggle/run_remote.py push <node>`, which is the only mode whose output can be
retrieved — an interactive session dies with the browser tab and `kernels output` 404s.

**Why not Wan 2.1** (tried first, 2026-07-25/26, six pushes): Wan is trained in **bfloat16**, and no
Kaggle free accelerator supports bf16 — T4 is Turing sm_75, P100 is Pascal sm_60, and bf16 arrives
with Ampere sm_80. In fp16 its activations overflow to NaN and every frame decodes to flat grey, at
29 minutes a shot. fp32 is numerically safe and ~8x slower, which cannot finish an episode inside a
12-hour session. AnimateDiff on SD1.5 is fp16-native, runs in minutes, and an anime checkpoint is a
better match for `style.md`'s flat cel-shaded look than a general-purpose video model.

**Hard-won details, each of which cost a run:**
- The repo is cloned to `/kaggle/tmp`, NOT `/kaggle/working`. Everything in `/kaggle/working` becomes
  the session's published output, and Kaggle caps how many files it indexes — a checkout there
  crowded the actual clips out of the output entirely.
- `machine_shape: NvidiaTeslaT4` is set in `kernel-metadata.json`. Without it the batch scheduler
  hands out a P100, which current torch ships no kernels for at all.
- The setup cell defines `transformers.utils.FLAX_WEIGHTS_NAME`, which the batch image's newer
  transformers removed and diffusers 0.33 imports at module load.
- Every clip's middle frame is checked for contrast before it is written, and the session ABORTS on a
  blank one. A numerically dead generation still writes a valid mp4 that passes every container,
  duration and audio check.

Output: `/kaggle/working/clips.zip`, re-zipped after every clip → feed to
`pipeline/render_t3.py <genome> <node> --clips <dir>`. Clips are short (3s at 8fps); `render_t3`
ping-pong-loops them to fill a beat, so a beat never shows a hard loop seam.

Provenance: every clip gets a `meta.yaml` (§7.2). The season's canon quality bar is decided by the
founder on material (R4/D8).


In [ ]:
# ---- config: what to render ----------------------------------------------
GENOME = "sapling"
NODE   = "001"        # any node id with a shots.md
BEATS  = [1,2,18]          # e.g. [1, 3] or None for all beats without status ✅
SEED   = 20260719      # fixed base seed: beat N renders with SEED + N (reproducible)
FPS_OUT = 7            # SVD is conditioned at 7fps
FPS    = 8             # AnimateDiff v1.5 motion module is trained at 8fps
STEPS  = 25            # 30 = faster/rougher, 50 = slower/cleaner
# SD1.5 anime checkpoints, tried in order — the first that loads anonymously wins.
# A gated repo returns 401 without a HuggingFace token, and a token is a
# credential (founder-reserved), so the notebook only ever uses open weights.
# Linaqruf/anything-v3.0 was the first choice and is gated.
BASES  = ["Lykon/dreamshaper-8",                          # illustrative, AnimateDiff-friendly
          "stable-diffusion-v1-5/stable-diffusion-v1-5",   # PROVEN to animate (spread 186)
          "gsdf/Counterfeit-V2.5"]                         # stills fine, does NOT animate
SVD    = "stabilityai/stable-video-diffusion-img2vid-xt"  # stage 2: real motion
STILL_W, STILL_H = 512, 768   # SD1.5 portrait it handles well; render_t3 crops to 9:16
MOTION = 160           # SVD motion_bucket_id: 20 = almost still, 180 = a lot.
                       # 127 gave MOTION 0.3 on a sparse still — SVD animates image
                       # CONTENT, so a subject on an empty ground has nothing to move.
SVD_FRAMES = 25        # 25 @ 7fps = ~3.5s of ACTUAL movement
IPADAPTER = True       # condition recurring characters on genomes/<g>/refs/*.png
IPA_SCALE = 0.35       # identity only. At 0.6 the reference dictated COMPOSITION too:
                       # four different beats — including a sprint and a close-up — all came
                       # back as the same centred standing figure in a circular vignette, and
                       # contrast dropped enough to trip the blank guard on two of them.
REPO_URL = "https://github.com/olegmlkvorg/banyan-city.git"


In [ ]:
# ---- setup: deps + repo (canon prompts come from shots.md, not a paste) ---
# Do NOT reinstall torch. Kaggle ships a working torch/torchvision pair; the
# first real run (2026-07-25) tried to pin its own and hit INTERNAL ASSERT
# FAILED in Dtype.cpp — a fresh torchvision against the already-imported
# torch. Install diffusers with --no-deps so pip cannot pull a second torch in
# behind it. WanPipeline needs diffusers >= 0.33 (0.32 lacks it entirely —
# that was the failure after the pin).
%pip -q install --no-deps "diffusers==0.33.1"
%pip -q install ftfy imageio imageio-ffmpeg pyyaml psutil

# A kernel that already imported an older diffusers keeps it in memory no
# matter what pip writes to disk (this bit the founder on 2026-07-25: the
# session still held 0.32.2). Compare disk vs memory and say so plainly.
import sys
from importlib.metadata import version
on_disk = version("diffusers")
in_mem = getattr(sys.modules.get("diffusers"), "__version__", None)
print(f"diffusers on disk: {on_disk}" + (f" | already imported in this kernel: {in_mem}" if in_mem else ""))
if in_mem and in_mem != on_disk:
    print("\n*** STOP: this kernel is holding an older diffusers.\n"
          "    Run > Restart & clear cell outputs, then Run All again.\n"
          "    (Nothing is lost — finished clips are skipped on re-run.) ***\n")


# Kaggle's BATCH image ships a newer transformers than its interactive one, and
# `transformers.utils.FLAX_WEIGHTS_NAME` is gone from it. diffusers 0.33 imports
# that name at module load, so `from diffusers import WanPipeline` died with
# "cannot import name 'FLAX_WEIGHTS_NAME'" before touching the GPU (first batch
# push, 2026-07-25). The names are plain filename constants and nothing on the
# Wan path reads a flax/tf checkpoint, so define what is missing rather than
# repinning transformers — a repin drags tokenizers and risks the torch pair.
# diffusers 0.33 also references transformers.CLIPFeatureExtractor, which current
# transformers renamed to CLIPImageProcessor. That is what made Lykon/dreamshaper-8
# — the anime-capable candidate — fail to load with "module transformers has no
# attribute CLIPFeatureExtractor", leaving only vanilla SD1.5, whose house style is
# watercolour rather than cel-shaded anime. Alias it.
import transformers as _tf
if not hasattr(_tf, "CLIPFeatureExtractor") and hasattr(_tf, "CLIPImageProcessor"):
    _tf.CLIPFeatureExtractor = _tf.CLIPImageProcessor
    print("aliased transformers.CLIPFeatureExtractor -> CLIPImageProcessor")

import transformers.utils as _tu
for _name, _val in (("FLAX_WEIGHTS_NAME", "flax_model.msgpack"),
                    ("TF2_WEIGHTS_NAME", "tf_model.h5"),
                    ("TF_WEIGHTS_NAME", "model.ckpt")):
    if not hasattr(_tu, _name):
        setattr(_tu, _name, _val)
        print(f"shimmed transformers.utils.{_name} (removed upstream)")

import pathlib
import subprocess
import sys

# Clone OUTSIDE /kaggle/working. Everything in /kaggle/working becomes the
# session's downloadable output, so cloning the repo there put the whole
# checkout — every committed episode mp4 included — into the output: the first
# successful run's `kernels output` was still pulling at 755 MB and had not
# reached the one clip that was actually rendered (2026-07-25). /kaggle/tmp is
# scratch and is not published.
CHECKOUT = pathlib.Path("/kaggle/tmp/banyan-city")
if not CHECKOUT.exists():
    CHECKOUT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CHECKOUT)], check=True)
sys.path.insert(0, str(CHECKOUT / "pipeline"))
from generate_shots import parse_shots
from sd_prompt import compress, extra_negatives  # CLIP stops at 77 tokens
import yaml

node_dirs = [d for d in (CHECKOUT / "genomes" / GENOME / "nodes").iterdir() if d.is_dir()]
node_dir = next((d for d in sorted(node_dirs) if d.name.startswith(NODE)), None)
assert node_dir, f"no node dir starting with {NODE!r} — check NODE above"
shots = parse_shots((node_dir / "shots.md").read_text())
todo = [s for s in shots if (BEATS is None and not s["done"]) or (BEATS and s["num"] in BEATS)]
print(f"{len(todo)} beat(s) to render for {node_dir.name}:")
for s in todo:
    print(f"  {s['num']:02d} {s['slug']}")

# the negative prompt is defined here so the model cell's bisect can use it too
NEG = ("photorealistic, 3d render, text, watermark, signature, low quality, blurry, "
       "extra limbs, deformed, jpeg artifacts, realistic skin texture")


In [ ]:
# ---- stage 1 model: SD1.5 makes the PICTURE ----------------------------------
# AnimateDiff was abandoned on the founder's verdict, 2026-07-26: "pretty much
# static, you could say. just cool looking static". He was right, and the number
# was already in front of me — frame-to-frame change measured 0.007-0.077, which I
# had read as "coherent video, not noise" without ever asking whether anything
# MOVED. The v1.5 motion module drifts; it does not animate. 16 frames at 8fps,
# ping-pong-looped to fill a five-second beat, is a shimmering still.
#
# So the job splits along the line of what actually works. SD1.5 makes a good
# still: measured contrast 57 on beat 1, on-style, reliable, and IP-Adapter keeps a
# character looking like himself. Stable Video Diffusion is image-to-video — hand
# it that still and it generates real camera and subject motion. It runs in fp16,
# so no bf16 wall, and it is free.
#
# One model resident at a time: all the stills first, then the SD pipeline is
# released and SVD is loaded. Two pipelines at once does not fit 14.6 GiB.
import gc

import numpy as np
import psutil
import torch
assert torch.cuda.is_available(), "No GPU: Settings > Accelerator = GPU (needs phone verification)"
_cap = torch.cuda.get_device_capability(0)
if f"sm_{_cap[0]}{_cap[1]}" not in torch.cuda.get_arch_list():
    raise SystemExit(f"{torch.cuda.get_device_name(0)} is sm_{_cap[0]}{_cap[1]}; this torch "
                     f"was built for {torch.cuda.get_arch_list()}. Use GPU T4 x2.")

from diffusers import DDIMScheduler, StableDiffusionPipeline
from PIL import Image

print(f"{torch.cuda.get_device_name(0)}: "
      f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB VRAM | "
      f"{psutil.virtual_memory().available / 2**30:.1f} GiB RAM")


def luma_spread(img):
    """LUMA spread 0-255. Must be luma: RGB percentiles are inflated by colour."""
    a = np.asarray(img, dtype=np.float32)
    if not np.isfinite(a).all():
        return 0.0
    if a.ndim == 3 and a.shape[-1] >= 3:
        a = a[..., 0] * 0.299 + a[..., 1] * 0.587 + a[..., 2] * 0.114
    lo, hi = np.percentile(a, 10), np.percentile(a, 90)
    return float(hi - lo) * (255.0 if a.max() <= 1.001 else 1.0)


def motion_of(frames):
    """Mean absolute frame-to-frame change, 0-255. The measure I failed to take.

    AnimateDiff's output scored near zero here while passing every contrast check I
    owned, which is exactly how "cool looking static" got shipped as success."""
    g = [np.asarray(f.convert("L"), dtype=np.float32) for f in frames]
    if len(g) < 2:
        return 0.0
    return float(np.mean([np.abs(g[i + 1] - g[i]).mean() for i in range(len(g) - 1)]))


sd = StableDiffusionPipeline.from_pretrained(
    BASES[0], torch_dtype=torch.float16, feature_extractor=None,
    safety_checker=None, requires_safety_checker=False)
sd.scheduler = DDIMScheduler.from_config(sd.scheduler.config, clip_sample=False,
                                        timestep_spacing="linspace", steps_offset=1)
sd.to("cuda")
BASE = BASES[0]

REFS = {}
if IPADAPTER:
    try:
        sd.load_ip_adapter("h94/IP-Adapter", subfolder="models",
                           weight_name="ip-adapter_sd15.bin")
        sd.set_ip_adapter_scale(IPA_SCALE)
        rdir = CHECKOUT / "genomes" / GENOME / "refs"
        for f in (sorted(rdir.glob("*.png")) if rdir.is_dir() else []):
            REFS[f.stem.lower()] = Image.open(f).convert("RGB").resize((512, 512))
        print(f"ip-adapter at {IPA_SCALE}; references: {sorted(REFS)}")
    except Exception as e:
        print(f"ip-adapter unavailable ({type(e).__name__}: {str(e)[:80]})")
        REFS = {}

WHO = {"jerry": ("goblin", "scavenger", "jerry")}

# Once an IP-Adapter is loaded, the UNet expects image embeds on EVERY call — pass
# none and it dies with "argument of type 'NoneType' is not iterable" deep inside
# the pipeline (2026-07-26, beat 1, which has no character in it). So a beat with no
# character gets a neutral grey image at scale 0, which conditions on nothing.
NEUTRAL = Image.new("RGB", (512, 512), (128, 128, 128))
gc.collect(); torch.cuda.empty_cache()
print(f"stage 1 ready — {BASE} at {STILL_W}x{STILL_H}")


In [ ]:
# ---- stage 1: a still per beat, then stage 2: SVD animates each one -----------
import shutil
import time
from datetime import date

NEG = ("photorealistic, 3d render, text, watermark, signature, low quality, blurry, "
       "extra limbs, deformed, jpeg artifacts, realistic skin texture")
out = pathlib.Path("/kaggle/working/clips"); out.mkdir(parents=True, exist_ok=True)
stills = pathlib.Path("/kaggle/working/stills"); stills.mkdir(parents=True, exist_ok=True)
blank, prompts = [], {}

# ---- stage 1 ----------------------------------------------------------------
for s in todo:
    png = stills / f"{s['num']:02d}-{s['slug']}.png"
    if png.exists():
        print(f"skip still {png.name}"); prompts[s["num"]] = compress(s["prompt"])[0]; continue
    t0 = time.time()
    ptext, dropped = compress(s["prompt"])
    prompts[s["num"]] = ptext
    extra = extra_negatives(s["prompt"])
    neg = f"{NEG}, {extra}" if extra else NEG
    ref = None
    for _n, _w in WHO.items():
        if _n in REFS and any(w in s["prompt"].lower() for w in _w):
            ref = REFS[_n]; break
    print(f"still {s['num']:02d} ({s['slug']}) …", flush=True)
    print(f"   {ptext}", flush=True)
    kw = {}
    if REFS:
        # adapter loaded => an image is mandatory; scale 0 makes it a no-op
        sd.set_ip_adapter_scale(IPA_SCALE if ref is not None else 0.0)
        kw["ip_adapter_image"] = ref if ref is not None else NEUTRAL
    img = sd(prompt=ptext, negative_prompt=neg, height=STILL_H, width=STILL_W,
             num_inference_steps=STEPS, guidance_scale=7.5,
             generator=torch.Generator(device="cpu").manual_seed(SEED + s["num"]),
             **kw).images[0]
    sp = luma_spread(img)
    img.save(png)
    print(f"   {png.name} in {(time.time()-t0)/60:.1f} min, contrast {sp:.0f}"
          + (f"  ref:{[n for n in WHO if REFS.get(n) is ref]}" if ref is not None else ""),
          flush=True)
    if sp < 35:
        blank.append((s["num"], s["slug"], round(sp)))
        print("   BLANK still — will still be animated, but flagged", flush=True)

del sd
gc.collect(); torch.cuda.empty_cache()
print(f"\nstage 1 done: {len(list(stills.glob('*.png')))} still(s)\n")

# ---- stage 2: real motion ---------------------------------------------------
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import export_to_video

svd = StableVideoDiffusionPipeline.from_pretrained(SVD, torch_dtype=torch.float16,
                                                  variant="fp16")
# SVD is the larger model; offload rather than hold it resident on a T4
svd.enable_model_cpu_offload()
# Ask before calling: SVD's VAE is AutoencoderKLTemporalDecoder, which has no
# enable_slicing (unlike the SD/SDXL VAEs). I feature-detected exactly this for
# Wan's VAE earlier today and then hardcoded the call here.
for _opt in ("enable_slicing", "enable_tiling"):
    _fn = getattr(svd.vae, _opt, None)
    if callable(_fn):
        _fn(); print(f"svd vae: {_opt}()")
print(f"stage 2 ready — {SVD}, motion_bucket={MOTION}, {SVD_FRAMES} frames @ {FPS_OUT}fps\n")

still_motion = []
for s in todo:
    png = stills / f"{s['num']:02d}-{s['slug']}.png"
    dest = out / f"{s['num']:02d}-{s['slug']}.mp4"
    if not png.exists() or dest.exists():
        continue
    t0 = time.time()
    print(f"animate {s['num']:02d} ({s['slug']}) …", flush=True)
    img = Image.open(png).convert("RGB")
    try:
        # SVD defaults to its training size, 1024x576 LANDSCAPE, and silently
        # ignores the aspect of the still you hand it — beat 1 came back widescreen
        # from a 512x768 portrait input (2026-07-26), and cropping that to 9:16
        # would throw away most of the frame. Ask for portrait explicitly.
        frames = svd(img, height=STILL_H, width=STILL_W,
                     num_frames=SVD_FRAMES, motion_bucket_id=MOTION,
                     noise_aug_strength=0.08, decode_chunk_size=4,   # more noise = more movement
                     generator=torch.Generator(device="cpu").manual_seed(SEED + s["num"])
                     ).frames[0]
    except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
        if not isinstance(e, torch.cuda.OutOfMemoryError) and \
           not any(k in str(e) for k in ("CUBLAS", "out of memory", "CUDA error")):
            raise
        print(f"   {type(e).__name__} — retrying with fewer frames", flush=True)
        gc.collect(); torch.cuda.empty_cache()
        frames = svd(img, height=STILL_H, width=STILL_W,
                     num_frames=14, motion_bucket_id=MOTION,
                     noise_aug_strength=0.02, decode_chunk_size=2,
                     generator=torch.Generator(device="cpu").manual_seed(SEED + s["num"])
                     ).frames[0]
    mo, sp = motion_of(frames), float(np.median([luma_spread(f) for f in frames[::4]]))
    export_to_video(frames, str(dest), fps=FPS_OUT)
    shutil.make_archive("/kaggle/working/clips", "zip", out)
    still_motion.append((s["num"], round(mo, 1)))
    print(f"   {dest.name} in {(time.time()-t0)/60:.1f} min, "
          f"contrast {sp:.0f}, MOTION {mo:.1f}", flush=True)
    dest.with_suffix(".meta.yaml").write_text(
        "# Shot provenance (\u00a77.2)\n" + yaml.safe_dump({
            "platform": "kaggle-free-gpu",
            "model": f"still: {BASE} (+IP-Adapter {IPA_SCALE}) | motion: {SVD}",
            "prompt": prompts.get(s["num"], ""), "negative_prompt": NEG,
            "seed": SEED + s["num"], "steps": STEPS,
            "frames": len(frames), "fps": FPS_OUT, "motion_bucket_id": MOTION,
            "measured_motion": round(mo, 2), "cost_usd": 0.00,
            "generated": str(date.today()),
        }, sort_keys=False))

# Motion is REPORTED, not gated: no threshold has been calibrated yet, and a
# mis-calibrated guard cost most of 2026-07-26. AnimateDiff scored near zero here.
if still_motion:
    print("\nmotion per beat (mean frame-to-frame change, 0-255):")
    for n, mo in still_motion:
        print(f"  beat {n:02d}: {mo}")
    print(f"  median {np.median([m for _, m in still_motion]):.1f} — "
          "AnimateDiff measured ~0.1-1.0 on the same beats")
if blank:
    print(f"\n{len(blank)} still(s) came out flat: {blank}")


In [ ]:
# ---- pack for download ------------------------------------------------------
import shutil
shutil.make_archive("/kaggle/working/clips", "zip", "/kaggle/working/clips")
print("download clips.zip from the Output tab, then locally:")
print(f"  python3 pipeline/render_t3.py {GENOME} {NODE} --clips <unzipped-dir> --out /tmp/{NODE}-episode.mp4")